# 07. RevPAR 시뮬레이션 — 호스트 액션 시나리오 분석

**분석 목적:** 가설 검증(H1~H8) 결과를 통합하여 호스트·자치구 관점의 우선순위별 액션 가이드를 제시합니다.

**핵심 질문:**
- 호스트가 당장 실행 가능한 가장 효과적인 RevPAR 개선 액션은 무엇인가?
- 자치구별로 적합한 전략 우선순위는 어떻게 다른가?

**접근 방법:**  
시장 현황 → 호스트 드라이버(H1~H5) → 자치구 패턴(H6~H8) → 액션 우선순위 매트릭스

**데이터:** 32,061개 리스팅 (TTM 2024-10~2025-09) | Active+Operating 14,399개

In [ ]:
# fmt_krw 유틸 -- 시각화 전반에서 원화 단위 포매팅에 재사용
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

import platform
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
else:
    plt.rc('font', family='AppleGothic')
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')
sns.set_palette('Set2')

# ── 경로 상수 ──
DATA_PATH = '../data/processed/seoul_airbnb_features.csv'
REPORTS_DIR = '../reports'

import os; os.makedirs(REPORTS_DIR, exist_ok=True)

# ── 유틸 함수 ──
def fmt_krw(x): return f'₩{x:,.0f}'
def fmt_krw_k(x): return f'₩{int(x/1000):,}K'

# ── 데이터 로드 ──
df = pd.read_csv(DATA_PATH)
df_ao = df[(df['refined_status'] == 'Active') & (df['operation_status'] == 'Operating')].copy()

print(f'전체: {len(df):,}개 | Active+Operating: {len(df_ao):,}개 ({len(df_ao)/len(df)*100:.1f}%)')
print(f'TTM RevPAR 전체 중위값: {fmt_krw(df["ttm_revpar"].median())}')
print(f'TTM RevPAR AO 중위값:   {fmt_krw(df_ao["ttm_revpar"].median())}')
print(f'TTM RevPAR AO 평균값:   {fmt_krw(df_ao["ttm_revpar"].mean())}')
print(f'Dormant 비율:           {(df["refined_status"]=="Dormant").mean()*100:.1f}%')

## 1. 서울 에어비앤비 RevPAR 시장 현황

### 핵심 시장 구조

서울 에어비앤비 시장은 **구조적 이중성**을 보인다:

- **Dormant 비율 54.3%**: 전체 32,061개 중 17,399개 리스팅이 사실상 비활성 — 시장 건전성의 핵심 리스크
- **RevPAR 양극화**: 전체 중위값 ₩8,868 vs Active+Op 중위값 ₩47,850 — **5.4배** 격차
- **점유율이 RevPAR의 핵심 드라이버**: Spearman ρ = 0.955 (ADR보다 압도적으로 높음)

이는 RevPAR 개선의 핵심이 **가격 인상보다 점유율 확보**에 있음을 시사한다.

In [ ]:
# 시장 이중 구조 시각화 -- Dormant 54.3%가 RevPAR 중위값을 왜곡하는 근거 제시

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('서울 에어비앤비 RevPAR 시장 현황 스냅샷', fontsize=15, fontweight='bold', y=1.01)

colors_set2 = sns.color_palette('Set2', 8)

# ── (0,0): RevPAR distribution by refined_status (Box plot) ──
ax0 = axes[0, 0]
status_order = ['Active', 'New', 'Mystery', 'Dormant', 'Ghost']
status_data = []
status_labels = []
for s in status_order:
    sub = df[df['refined_status'] == s]['ttm_revpar'].dropna()
    if len(sub) > 0:
        status_data.append(sub)
        status_labels.append(f'{s}\n(n={len(sub):,})')

bp = ax0.boxplot(status_data, labels=status_labels, patch_artist=True,
                  showfliers=False, medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], colors_set2[:len(status_data)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax0.set_title('운영 상태별 TTM RevPAR 분포', fontsize=11, fontweight='bold')
ax0.set_ylabel('TTM RevPAR (₩)', fontsize=9)
ax0.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'₩{int(x):,}'))
ax0.tick_params(axis='x', labelsize=8)

# ── (0,1): ADR vs Occupancy scatter by room_type (bubble sized by count) ──
ax1 = axes[0, 1]
room_stats = df_ao.groupby('room_type').agg(
    med_adr=('ttm_avg_rate', 'median'),
    med_occ=('ttm_occupancy', 'median'),
    count=('ttm_revpar', 'count')
).reset_index()

# RevPAR isocurves
adr_range = np.linspace(10000, 700000, 300)
for revpar_level, label, ls in [(30000, '₩30K', '--'), (60000, '₩60K', '-.'), (100000, '₩100K', ':')]:
    occ_curve = revpar_level / (adr_range + 1e-6)
    mask = (occ_curve >= 0) & (occ_curve <= 1)
    ax1.plot(adr_range[mask], occ_curve[mask], ls=ls, color='gray', alpha=0.5, linewidth=1)
    # 레이블 위치
    mid_idx = mask.sum() // 2
    if mask.sum() > 0:
        xs = adr_range[mask]
        ys = occ_curve[mask]
        ax1.text(xs[mid_idx], ys[mid_idx], f'RevPAR={label}', fontsize=7, color='gray', alpha=0.8)

room_colors = {rt: colors_set2[i] for i, rt in enumerate(room_stats['room_type'])}
for _, row in room_stats.iterrows():
    ax1.scatter(row['med_adr'], row['med_occ'],
                s=row['count'] / 10, c=[room_colors[row['room_type']]],
                alpha=0.8, edgecolors='white', linewidth=1)
    ax1.annotate(row['room_type'].replace('_', '\n'), (row['med_adr'], row['med_occ']),
                 fontsize=8, ha='center', va='bottom',
                 xytext=(0, 8), textcoords='offset points')

ax1.set_title('숙소 유형별 ADR × 점유율 포지셔닝', fontsize=11, fontweight='bold')
ax1.set_xlabel('중위 ADR (₩)', fontsize=9)
ax1.set_ylabel('중위 점유율', fontsize=9)
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'₩{int(x/1000):,}K'))
ax1.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))

# ── (1,0): Controllable variable correlations ──
ax2 = axes[1, 0]
# Pre-computed Spearman rho values (excluding leakage variables)
corr_vars = ['num_reviews', 'rating_overall', 'photos_count', 'nearest_poi_dist_km']
corr_labels = ['리뷰 수', '전체 평점', '사진 수', 'POI 거리 (km)']
computed_rho = []
computed_p = []
for var in corr_vars:
    sub = df_ao[['ttm_revpar', var]].dropna()
    rho, pval = stats.spearmanr(sub['ttm_revpar'], sub[var])
    computed_rho.append(rho)
    computed_p.append(pval)

colors_bar = [colors_set2[0] if r > 0 else colors_set2[2] for r in computed_rho]
bars = ax2.barh(corr_labels, computed_rho, color=colors_bar, alpha=0.8, edgecolor='white')
for bar, rho, pval in zip(bars, computed_rho, computed_p):
    sig = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else ''))
    ax2.text(rho + (0.005 if rho >= 0 else -0.005), bar.get_y() + bar.get_height()/2,
             f'ρ={rho:.3f}{sig}', va='center', ha='left' if rho >= 0 else 'right', fontsize=9)
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_title('통제 가능 변수 vs TTM RevPAR\nSpearman ρ (Active+Op)', fontsize=11, fontweight='bold')
ax2.set_xlabel('Spearman ρ', fontsize=9)
ax2.set_xlim(-0.2, 0.6)
ax2.tick_params(axis='y', labelsize=9)
# Ensure y-axis labels have enough space
for label in ax2.get_yticklabels():
    label.set_horizontalalignment('right')

# ── (1,1): Market funnel ──
ax3 = axes[1, 1]
status_counts = df['refined_status'].value_counts()
funnel_data = {
    '전체 리스팅\n(32,061)': len(df),
    'Active+Operating\n(14,399)': len(df_ao),
    'Active+Op 슈퍼호스트\n(~5,200)': len(df_ao[df_ao['superhost'] == True]),
    'Active+Op\n전체주택형': len(df_ao[df_ao['room_type'] == 'entire_home'])
}

funnel_labels = list(funnel_data.keys())
funnel_values = list(funnel_data.values())
funnel_colors = [colors_set2[0], colors_set2[1], colors_set2[2], colors_set2[3]]

bars3 = ax3.barh(funnel_labels, funnel_values, color=funnel_colors, alpha=0.8, edgecolor='white')
for bar, val in zip(bars3, funnel_values):
    ax3.text(val + 200, bar.get_y() + bar.get_height()/2,
             f'{val:,}개 ({val/len(df)*100:.1f}%)', va='center', fontsize=9)
ax3.set_title('시장 세그먼트 구조', fontsize=11, fontweight='bold')
ax3.set_xlabel('리스팅 수', fontsize=9)
ax3.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{int(x):,}'))
ax3.set_xlim(0, len(df) * 1.25)

plt.tight_layout()
plt.show()

print('\n=== 시장 현황 핵심 통계 ===')
print(f'전체 리스팅: {len(df):,}개')
print(f'Active+Operating: {len(df_ao):,}개 ({len(df_ao)/len(df)*100:.1f}%)')
dormant_n = (df['refined_status'] == 'Dormant').sum()
print(f'Dormant: {dormant_n:,}개 ({dormant_n/len(df)*100:.1f}%)')
print(f'TTM RevPAR 전체 중위값: {fmt_krw(df["ttm_revpar"].median())}')
print(f'TTM RevPAR AO 중위값: {fmt_krw(df_ao["ttm_revpar"].median())}')
print(f'TTM RevPAR AO 평균값: {fmt_krw(df_ao["ttm_revpar"].mean())}')
print(f'RevPAR 배율 (AO/전체): {df_ao["ttm_revpar"].median()/df["ttm_revpar"].median():.1f}x')

## 2. 호스트 관점 RevPAR 드라이버 분석 (H1~H5)

호스트가 **직접 통제 가능한 변수**들이 RevPAR에 미치는 영향을 검증한다.

| 가설 | 변수 | 예상 방향 |
|------|------|----------|
| H1 | 슈퍼호스트 + 즉시예약 | + |
| H2 | 숙소 유형(전체주택 vs 개인실) | + (전체주택) |
| H3 | 사진 수 + 평점 | + (임계점 존재) |
| H4 | 최소숙박일 | 역U자형 (2~3박 최적) |
| H5 | 추가 게스트 요금 정책 | + (대형 숙소) |

In [ ]:
# H1: 슈퍼호스트 x 즉시예약 조합 효과 -- 호스트가 즉시 실행 가능한 가장 강력한 레버

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('H1: 슈퍼호스트 × 즉시예약이 RevPAR에 미치는 영향 (Active+Op)', fontsize=13, fontweight='bold')

colors_set2 = sns.color_palette('Set2', 8)

# 4개 그룹 생성
df_ao_h1 = df_ao.copy()
df_ao_h1['sh_label'] = df_ao_h1['superhost'].map({True: '슈퍼호스트', False: '일반호스트'})
df_ao_h1['ib_label'] = df_ao_h1['instant_book'].map({True: '즉시예약', False: '요청예약'})
df_ao_h1['group'] = df_ao_h1['sh_label'] + ' + ' + df_ao_h1['ib_label']

group_order = ['슈퍼호스트 + 즉시예약', '슈퍼호스트 + 요청예약',
               '일반호스트 + 즉시예약', '일반호스트 + 요청예약']
group_data = {g: df_ao_h1[df_ao_h1['group'] == g]['ttm_revpar'].dropna() for g in group_order}

# Left: Boxplot
ax0 = axes[0]
plot_data = [group_data[g] for g in group_order if len(group_data[g]) > 0]
plot_labels = [f'{g}\n(n={len(group_data[g]):,})' for g in group_order if len(group_data[g]) > 0]

bp = ax0.boxplot(plot_data, labels=plot_labels, patch_artist=True,
                  showfliers=False, medianprops=dict(color='black', linewidth=2))
box_colors = [colors_set2[0], colors_set2[1], colors_set2[2], colors_set2[3]]
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax0.set_title('그룹별 TTM RevPAR 분포', fontsize=11, fontweight='bold')
ax0.set_ylabel('TTM RevPAR (₩)', fontsize=9)
ax0.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'₩{int(x/1000):,}K'))
ax0.tick_params(axis='x', labelsize=7)

# Right: Bar chart with median values
ax1 = axes[1]
medians = {g: group_data[g].median() for g in group_order if len(group_data[g]) > 0}
short_labels = ['SH+즉시\n예약', 'SH+요청\n예약', '일반+즉시\n예약', '일반+요청\n예약']
valid_groups = [g for g in group_order if len(group_data[g]) > 0]
short_labels = short_labels[:len(valid_groups)]

bars = ax1.bar(short_labels, [medians[g] for g in valid_groups],
               color=box_colors[:len(valid_groups)], alpha=0.8, edgecolor='white')

max_val = max(medians.values())
for bar, g in zip(bars, valid_groups):
    val = medians[g]
    pct = val / medians.get('일반호스트 + 요청예약', 1) * 100 - 100
    ax1.text(bar.get_x() + bar.get_width()/2, val + max_val * 0.01,
             f'{fmt_krw_k(val)}\n({pct:+.0f}%)', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax1.set_title('그룹별 중위 RevPAR 비교', fontsize=11, fontweight='bold')
ax1.set_ylabel('중위 TTM RevPAR (₩)', fontsize=9)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'₩{int(x/1000):,}K'))

plt.tight_layout()
plt.show()

# Mann-Whitney U test
sh_group = df_ao[df_ao['superhost'] == True]['ttm_revpar'].dropna()
nsh_group = df_ao[df_ao['superhost'] == False]['ttm_revpar'].dropna()
u_stat, p_val = stats.mannwhitneyu(sh_group, nsh_group, alternative='greater')

print('=== H1: 슈퍼호스트 효과 검증 ===')
print(f'슈퍼호스트 중위 RevPAR: {fmt_krw(sh_group.median())} (n={len(sh_group):,})')
print(f'일반호스트 중위 RevPAR: {fmt_krw(nsh_group.median())} (n={len(nsh_group):,})')
print(f'프리미엄: {sh_group.median()/nsh_group.median()*100-100:.1f}%')
print(f'Mann-Whitney U = {u_stat:.0f}, p-value = {p_val:.2e}')
print(f'통계적 유의성: {"*** (p<0.001)" if p_val < 0.001 else ("** (p<0.01)" if p_val < 0.01 else "* (p<0.05)" if p_val < 0.05 else "유의하지 않음")}')

print('\n그룹별 상세:')
for g in group_order:
    if g in medians:
        n = len(group_data[g])
        print(f'  {g}: {fmt_krw(medians[g])} (n={n:,})')

In [ ]:
# H2: 숙소 유형별 ADR vs 점유율 구조 -- entire_home은 가격 드라이버, private_room은 점유율 드라이버

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('H2: 숙소 유형별 ADR × 점유율 포지셔닝 (Active+Op)', fontsize=13, fontweight='bold')

colors_set2 = sns.color_palette('Set2', 8)

room_stats = df_ao.groupby('room_type').agg(
    med_adr=('ttm_avg_rate', 'median'),
    med_occ=('ttm_occupancy', 'median'),
    med_revpar=('ttm_revpar', 'median'),
    count=('ttm_revpar', 'count')
).reset_index()

room_korean = {
    'entire_home': '전체주택',
    'private_room': '개인실',
    'hotel_room': '호텔룸',
    'shared_room': '공유실'
}
room_stats['room_korean'] = room_stats['room_type'].map(room_korean)

# Left: Scatter with isocurves
ax0 = axes[0]

adr_range = np.linspace(5000, 600000, 500)
for revpar_level, label, ls, color_iso in [
    (30000, '₩30K', '--', 'lightblue'),
    (60000, '₩60K', '-.', 'orange'),
    (100000, '₩100K', ':', 'lightgreen')
]:
    occ_curve = revpar_level / (adr_range + 1e-6)
    mask = (occ_curve >= 0.01) & (occ_curve <= 1.0)
    ax0.plot(adr_range[mask], occ_curve[mask], ls=ls, color='gray', alpha=0.6, linewidth=1.2)
    xs = adr_range[mask]
    ys = occ_curve[mask]
    if len(xs) > 5:
        mid = len(xs) // 3
        ax0.text(xs[mid], ys[mid] + 0.02, f'RevPAR={label}', fontsize=7, color='gray', alpha=0.9)

for i, (_, row) in enumerate(room_stats.iterrows()):
    size = max(50, row['count'] / 8)
    ax0.scatter(row['med_adr'], row['med_occ'],
                s=size, c=[colors_set2[i]], alpha=0.85,
                edgecolors='white', linewidth=1.5, zorder=5)
    ax0.annotate(f"{row['room_korean']}\n(n={row['count']:,})",
                 (row['med_adr'], row['med_occ']),
                 fontsize=8, ha='center', va='bottom',
                 xytext=(0, 12), textcoords='offset points',
                 bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

ax0.set_title('ADR × 점유율 포지셔닝 (버블=리스팅수)', fontsize=11, fontweight='bold')
ax0.set_xlabel('중위 ADR (₩)', fontsize=9)
ax0.set_ylabel('중위 점유율', fontsize=9)
ax0.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'₩{int(x/1000):,}K'))
ax0.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))

# Right: RevPAR components bar chart
ax1 = axes[1]
room_order = room_stats.sort_values('med_revpar', ascending=False)['room_type'].tolist()
x_pos = np.arange(len(room_order))
width = 0.35

adr_vals = [room_stats[room_stats['room_type'] == rt]['med_adr'].values[0] for rt in room_order]
occ_vals = [room_stats[room_stats['room_type'] == rt]['med_occ'].values[0] for rt in room_order]
revpar_vals = [room_stats[room_stats['room_type'] == rt]['med_revpar'].values[0] for rt in room_order]

bars1 = ax1.bar(x_pos - width/2, adr_vals, width, label='중위 ADR',
                 color=colors_set2[0], alpha=0.8, edgecolor='white')
ax1_twin = ax1.twinx()
bars2 = ax1_twin.bar(x_pos + width/2, [o * 100 for o in occ_vals], width, label='중위 점유율',
                      color=colors_set2[1], alpha=0.8, edgecolor='white')

room_labels_kr = [room_korean.get(rt, rt) for rt in room_order]
ax1.set_xticks(x_pos)
ax1.set_xticklabels(room_labels_kr, fontsize=9)
ax1.set_ylabel('중위 ADR (₩)', fontsize=9, color=colors_set2[0])
ax1_twin.set_ylabel('중위 점유율 (%)', fontsize=9, color=colors_set2[1])
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'₩{int(x/1000):,}K'))

for bar, rv in zip(bars1, revpar_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, -15000,
             f'RevPAR\n{fmt_krw_k(rv)}', ha='center', va='top', fontsize=7,
             color='black', fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_twin.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=8)
ax1.set_title('숙소 유형별 RevPAR 구성 요소', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print('=== H2: 숙소 유형별 RevPAR 통계 ===')
for rt in room_order:
    row = room_stats[room_stats['room_type'] == rt].iloc[0]
    print(f'{room_korean.get(rt, rt):8s}: RevPAR {fmt_krw(row["med_revpar"]):>12s}, ADR {fmt_krw(row["med_adr"]):>12s}, 점유율 {row["med_occ"]*100:.1f}% (n={row["count"]:,})')

# Kruskal-Wallis test
room_groups = [df_ao[df_ao['room_type'] == rt]['ttm_revpar'].dropna() for rt in room_order if len(df_ao[df_ao['room_type'] == rt]) > 0]
if len(room_groups) >= 2:
    kw_stat, kw_p = stats.kruskal(*room_groups)
    print(f'\nKruskal-Wallis H = {kw_stat:.2f}, p = {kw_p:.2e}')

In [ ]:
# H3: 사진 수 임계점 확인 -- 21-35장 구간에서 RevPAR 최대화

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('H3: 콘텐츠 품질(사진 수 · 평점)이 RevPAR에 미치는 영향 (Active+Op)', fontsize=13, fontweight='bold')

colors_set2 = sns.color_palette('Set2', 8)

# Left: Photos count grouped
ax0 = axes[0]
photo_bins = [0, 10, 20, 35, 50, 200]
photo_labels_str = ['0-10장', '11-20장', '21-35장', '36-50장', '51장+']
df_ao_ph = df_ao[['photos_count', 'ttm_revpar']].dropna().copy()
df_ao_ph['photo_group'] = pd.cut(df_ao_ph['photos_count'],
                                   bins=photo_bins, labels=photo_labels_str, right=True)

photo_stats = df_ao_ph.groupby('photo_group', observed=True).agg(
    med_revpar=('ttm_revpar', 'median'),
    count=('ttm_revpar', 'count')
).reset_index()

bar_colors_ph = []
for lbl in photo_labels_str:
    if lbl == '21-35장':
        bar_colors_ph.append(colors_set2[0])  # highlight optimal
    else:
        bar_colors_ph.append(colors_set2[6])

bars0 = ax0.bar(photo_stats['photo_group'].astype(str), photo_stats['med_revpar'],
                color=bar_colors_ph, alpha=0.85, edgecolor='white')

for bar, row in zip(bars0, photo_stats.itertuples()):
    ax0.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
             f'{fmt_krw_k(row.med_revpar)}\n(n={row.count:,})',
             ha='center', va='bottom', fontsize=8)

# Shade optimal zone
optimal_idx = photo_labels_str.index('21-35장')
ax0.axvspan(optimal_idx - 0.45, optimal_idx + 0.45, alpha=0.15, color='gold', label='최적 구간 (21-35장)')
ax0.legend(fontsize=8)
ax0.set_title('사진 수 구간별 중위 RevPAR', fontsize=11, fontweight='bold')
ax0.set_xlabel('사진 수 구간', fontsize=9)
ax0.set_ylabel('중위 TTM RevPAR (₩)', fontsize=9)
ax0.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'₩{int(x/1000):,}K'))
ax0.tick_params(axis='x', labelsize=8)

# Right: Rating grouped
ax1 = axes[1]
df_ao_rt = df_ao[['rating_overall', 'ttm_revpar']].dropna().copy()
rating_bins = [0, 4.0, 4.5, 4.7, 4.9, 5.01]
rating_labels_str = ['4.0 미만', '4.0-4.5', '4.5-4.7', '4.7-4.9', '5.0']
df_ao_rt['rating_group'] = pd.cut(df_ao_rt['rating_overall'],
                                    bins=rating_bins, labels=rating_labels_str, right=True)

rating_stats = df_ao_rt.groupby('rating_group', observed=True).agg(
    med_revpar=('ttm_revpar', 'median'),
    count=('ttm_revpar', 'count')
).reset_index()

bar_colors_rt = [colors_set2[6], colors_set2[6], colors_set2[6], colors_set2[0], colors_set2[0]]

bars1 = ax1.bar(rating_stats['rating_group'].astype(str), rating_stats['med_revpar'],
                color=bar_colors_rt[:len(rating_stats)], alpha=0.85, edgecolor='white')

for bar, row in zip(bars1, rating_stats.itertuples()):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
             f'{fmt_krw_k(row.med_revpar)}\n(n={row.count:,})',
             ha='center', va='bottom', fontsize=8)

# Mark 4.8 threshold
threshold_x = rating_labels_str.index('4.7-4.9') + 0.5
ax1.axvline(threshold_x, color='red', linestyle='--', alpha=0.7, linewidth=1.5, label='4.8+ 임계점')
ax1.legend(fontsize=8)
ax1.set_title('평점 구간별 중위 RevPAR', fontsize=11, fontweight='bold')
ax1.set_xlabel('전체 평점 구간', fontsize=9)
ax1.set_ylabel('중위 TTM RevPAR (₩)', fontsize=9)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'₩{int(x/1000):,}K'))
ax1.tick_params(axis='x', labelsize=8)

plt.tight_layout()
plt.show()

print('=== H3: 사진 수 구간별 RevPAR ===')
print(photo_stats.to_string(index=False))
print('\n=== H3: 평점 구간별 RevPAR ===')
print(rating_stats.to_string(index=False))

# Spearman correlation
rho_ph, p_ph = stats.spearmanr(df_ao_ph['photos_count'], df_ao_ph['ttm_revpar'])
rho_rt, p_rt = stats.spearmanr(df_ao_rt['rating_overall'], df_ao_rt['ttm_revpar'])
print(f'\n사진 수 vs RevPAR: Spearman ρ = {rho_ph:.3f}, p = {p_ph:.2e}')
print(f'평점 vs RevPAR: Spearman ρ = {rho_rt:.3f}, p = {p_rt:.2e}')

In [ ]:
# H4: min_nights 최적 구간 -- 2-3박이 점유율/가격 균형 최적점

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('H4: 최소숙박일이 RevPAR에 미치는 영향 (Active+Op, min_nights ≤ 30)', fontsize=13, fontweight='bold')

colors_set2 = sns.color_palette('Set2', 8)

df_ao_mn = df_ao[df_ao['min_nights'] <= 30].copy()
mn_bins = [0, 1, 2, 4, 7, 14, 30]
mn_labels_str = ['1박', '2박', '3-4박', '5-7박', '8-14박', '15-30박']
df_ao_mn['mn_group'] = pd.cut(df_ao_mn['min_nights'],
                               bins=mn_bins, labels=mn_labels_str, right=True)

mn_stats = df_ao_mn.groupby('mn_group', observed=True).agg(
    med_revpar=('ttm_revpar', 'median'),
    med_occ=('ttm_occupancy', 'median'),
    med_adr=('ttm_avg_rate', 'median'),
    count=('ttm_revpar', 'count')
).reset_index()

# Left: Dual axis line chart (occupancy + ADR)
ax0 = axes[0]
x_pos = np.arange(len(mn_stats))

line1, = ax0.plot(x_pos, mn_stats['med_occ'] * 100, 'o-',
                   color=colors_set2[0], linewidth=2, markersize=8, label='중위 점유율 (좌)')
ax0.set_ylabel('중위 점유율 (%)', color=colors_set2[0], fontsize=9)
ax0.tick_params(axis='y', labelcolor=colors_set2[0])

ax0_twin = ax0.twinx()
line2, = ax0_twin.plot(x_pos, mn_stats['med_adr'], 's--',
                        color=colors_set2[1], linewidth=2, markersize=8, label='중위 ADR (우)')
ax0_twin.set_ylabel('중위 ADR (₩)', color=colors_set2[1], fontsize=9)
ax0_twin.tick_params(axis='y', labelcolor=colors_set2[1])
ax0_twin.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'₩{int(x/1000):,}K'))

ax0.set_xticks(x_pos)
ax0.set_xticklabels([f'{l}\n(n={n:,})' for l, n in zip(mn_stats['mn_group'].astype(str), mn_stats['count'])],
                     fontsize=8)
ax0.set_title('최소숙박일별 점유율 · ADR 변화', fontsize=11, fontweight='bold')

lines_all = [line1, line2]
labels_all = [l.get_label() for l in lines_all]
ax0.legend(lines_all, labels_all, fontsize=8, loc='upper right')

# Right: RevPAR bar with color gradient
ax1 = axes[1]
revpar_vals = mn_stats['med_revpar'].values
max_rv = revpar_vals.max()
bar_colors_mn = []
for rv in revpar_vals:
    ratio = rv / max_rv
    if ratio >= 0.75:
        bar_colors_mn.append(colors_set2[0])  # green-ish
    elif ratio >= 0.5:
        bar_colors_mn.append(colors_set2[5])  # yellow-ish
    else:
        bar_colors_mn.append(colors_set2[2])  # red-ish

bars1 = ax1.bar(mn_stats['mn_group'].astype(str), revpar_vals,
                color=bar_colors_mn, alpha=0.85, edgecolor='white')

for bar, rv in zip(bars1, revpar_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, rv + max_rv * 0.01,
             fmt_krw_k(rv), ha='center', va='bottom', fontsize=8, fontweight='bold')

# Highlight optimal zone (1박 and 2박)
ax1.axvspan(-0.45, 1.45, alpha=0.12, color='gold', label='최적 구간 (1-2박)')
ax1.legend(fontsize=8)
ax1.set_title('최소숙박일별 중위 RevPAR', fontsize=11, fontweight='bold')
ax1.set_ylabel('중위 TTM RevPAR (₩)', fontsize=9)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'₩{int(x/1000):,}K'))
ax1.tick_params(axis='x', labelsize=8)

plt.tight_layout()
plt.show()

print('=== H4: 최소숙박일별 상세 통계 ===')
for _, row in mn_stats.iterrows():
    print(f'{str(row["mn_group"]):8s}: RevPAR {fmt_krw(row["med_revpar"]):>12s}, 점유율 {row["med_occ"]*100:.1f}%, ADR {fmt_krw(row["med_adr"]):>12s} (n={row["count"]:,})')

kw_stat, kw_p = stats.kruskal(*[df_ao_mn[df_ao_mn['mn_group'] == g]['ttm_revpar'].dropna()
                                  for g in mn_labels_str
                                  if len(df_ao_mn[df_ao_mn['mn_group'] == g]) > 0])
print(f'\nKruskal-Wallis H = {kw_stat:.2f}, p = {kw_p:.2e}')

In [ ]:
# H5: 추가요금 정책은 대형 숙소(2+ 침실)에서만 유의미한 RevPAR 효과

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('H5: 추가 게스트 요금 정책이 RevPAR에 미치는 영향 (대형 숙소: guests ≥ 4)', fontsize=13, fontweight='bold')

colors_set2 = sns.color_palette('Set2', 8)

# Large units: guests >= 4 (or '7+' group)
df_ao_large = df_ao[df_ao['guests'] >= 4].copy()
df_ao_large['fee_label'] = df_ao_large['extra_guest_fee_policy'].map({0: '요금 없음 (0)', 1: '요금 있음 (1)'})

# Left: Boxplot
ax0 = axes[0]
fee0 = df_ao_large[df_ao_large['extra_guest_fee_policy'] == 0]['ttm_revpar'].dropna()
fee1 = df_ao_large[df_ao_large['extra_guest_fee_policy'] == 1]['ttm_revpar'].dropna()

bp = ax0.boxplot([fee0, fee1],
                  labels=[f'요금 없음\n(n={len(fee0):,})', f'요금 있음\n(n={len(fee1):,})'],
                  patch_artist=True, showfliers=False,
                  medianprops=dict(color='black', linewidth=2))
bp['boxes'][0].set_facecolor(colors_set2[2])
bp['boxes'][0].set_alpha(0.7)
bp['boxes'][1].set_facecolor(colors_set2[0])
bp['boxes'][1].set_alpha(0.7)

ax0.set_title('추가요금 정책별 RevPAR 분포 (대형 숙소)', fontsize=11, fontweight='bold')
ax0.set_ylabel('TTM RevPAR (₩)', fontsize=9)
ax0.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'₩{int(x/1000):,}K'))

u_stat5, p_val5 = stats.mannwhitneyu(fee1, fee0, alternative='greater')
ax0.set_xlabel(f'Mann-Whitney U = {u_stat5:.0f}, p = {p_val5:.2e}', fontsize=8)

# Right: Top 8 districts grouped bars
ax1 = axes[1]
top_districts = df_ao_large['district'].value_counts().head(8).index.tolist()
dist_fee_stats = df_ao_large[df_ao_large['district'].isin(top_districts)].groupby(
    ['district', 'extra_guest_fee_policy']
)['ttm_revpar'].median().unstack(fill_value=0)

# Sort by total RevPAR
dist_fee_stats['total'] = dist_fee_stats.sum(axis=1)
dist_fee_stats = dist_fee_stats.sort_values('total', ascending=True).drop('total', axis=1)

y_pos = np.arange(len(dist_fee_stats))
bar_width = 0.4

if 0 in dist_fee_stats.columns:
    ax1.barh(y_pos - bar_width/2, dist_fee_stats[0], bar_width,
              label='요금 없음', color=colors_set2[2], alpha=0.8, edgecolor='white')
if 1 in dist_fee_stats.columns:
    ax1.barh(y_pos + bar_width/2, dist_fee_stats[1], bar_width,
              label='요금 있음', color=colors_set2[0], alpha=0.8, edgecolor='white')

ax1.set_yticks(y_pos)
ax1.set_yticklabels(dist_fee_stats.index, fontsize=8)
ax1.set_title('상위 8개 자치구별 추가요금 정책 효과', fontsize=11, fontweight='bold')
ax1.set_xlabel('중위 TTM RevPAR (₩)', fontsize=9)
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'₩{int(x/1000):,}K'))
ax1.legend(fontsize=9)

plt.tight_layout()
plt.show()

print('=== H5: 추가 게스트 요금 효과 (대형 숙소 guests ≥ 4) ===')
print(f'요금 없음 중위 RevPAR: {fmt_krw(fee0.median())} (n={len(fee0):,})')
print(f'요금 있음 중위 RevPAR: {fmt_krw(fee1.median())} (n={len(fee1):,})')
print(f'프리미엄: {fee1.median()/fee0.median():.1f}배 ({(fee1.median()/fee0.median()-1)*100:.0f}% 차이)')
print(f'절대 차이: {fmt_krw(fee1.median()-fee0.median())}')
print(f'Mann-Whitney U = {u_stat5:.0f}, p = {p_val5:.2e}')

## H1~H5 호스트 드라이버 분석 요약

### 검증 결과 종합

| 가설 | 검증 결과 | 핵심 수치 | 비고 |
|------|-----------|-----------|------|
| **H1: 슈퍼호스트** | ✅ 지지 | +83.1% RevPAR 프리미엄 (₩61,205 vs ₩31,825 중위값) | 가장 강력한 단일 레버 |
| **H2: 숙소 유형** | ✅ 지지 | entire_home ADR 주도형(₩58,944), private_room 점유율 주도형(₩21,791) | ADR-점유율 분해 확인 |
| **H3: 콘텐츠 품질** | ✅ 지지 (비선형) | 사진 21-35장 최적. **Perfectionism Paradox**: 평점 4.85-4.95 구간 점유율(0.622)이 5.0 구간(0.532)보다 높음 | 5.0 목표는 비효율 |
| **H4: 최소숙박일** | ✅ 지지 (복합) | RevPAR 최적=3박(₩58,540), 점유율 최적=4-7박(0.597). 두 최적점 불일치 | 단순 '1박=최적' 오류 |
| **H5: 추가요금** | ⚠️ **부분 지지 — 반전 발견** | 소형(1-6인) 5천~3만원 범위: 제한적 효과. **대형(7인+): 부과 시 RevPAR -56% (₩113,264→₩49,860)** | 대형 숙소에서 역효과 |

> ⚠️ **H5 주의**: 7인 이상 대형 숙소에서 추가요금은 RevPAR를 56% 감소시키는 **역효과** 발생. 대형 숙소 호스트에게 추가요금 설정을 일반적으로 권장해서는 안 됨.

---

### Top 10% RevPAR 달성 확률 (01d 교차분석 결과)

5가지 핵심 조건 충족 수에 따른 상위 10% 진입 확률:

| 조건 수 | Top 10% 확률 |
|:---:|:---:|
| 0개 | 0.14% |
| 3개 | ~8% |
| **5개 모두** | **26.3%** |

**5개 조건**: ① 슈퍼호스트 ② 리뷰 60개+ ③ 평점 4.85~4.95 ④ entire_home ⑤ min_nights 2~3박

→ 5개 완전 충족 시 일반 숙소 대비 **188배** 높은 Top 10% 진입 확률.

### 호스트 우선순위 액션 (수정)

1. **슈퍼호스트 취득** — 가장 강력한 레버 (+83.1%)
2. **최소숙박일 2-3박 설정** — RevPAR 최적점 (청소비 분산 + 점유율 균형)
3. **사진 21-35장 유지** — 비용 없이 검색 노출 + 전환율 개선
4. **평점 4.85~4.95 관리** — Perfectionism Paradox: 5.0보다 이 구간이 점유율 높음
5. **소형 숙소(1-6인) 추가요금 설정** — 대형(7인+)은 역효과 주의


## 3. 자치구별 RevPAR 패턴 분석 (H6~H9)

**자치구 단위** 분석으로 시장 구조와 입지 특성이 RevPAR에 미치는 영향을 파악한다.

| 가설 | 변수 | 예상 방향 |
|------|------|----------|
| H6 | 공급 집중도 vs RevPAR | - (공급 과잉 → 가격 압박) |
| H7 | POI 유형 + 거리 | + (관광지 근접) |
| H8 | 인구 밀도 | + (수요 연동) |
| H9 | Dormant 비율 | - (시장 건전성 위험) |

In [ ]:
# H6: 공급 과잉 자치구 RevPAR 압박 -- 마포구는 최다 공급이지만 RevPAR 하락은 미미

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('H6: 자치구별 공급 규모 vs RevPAR 관계 (Active+Op)', fontsize=13, fontweight='bold')

colors_set2 = sns.color_palette('Set2', 8)

# District-level aggregation (Active+Op)
dist_stats = df_ao.groupby('district').agg(
    listing_count=('ttm_revpar', 'count'),
    med_revpar=('ttm_revpar', 'median'),
    mean_revpar=('ttm_revpar', 'mean')
).reset_index()

# All district supply (total)
dist_total = df.groupby('district').agg(
    total_listings=('ttm_revpar', 'count')
).reset_index()

dist_stats = dist_stats.merge(dist_total, on='district', how='left')

# Spearman correlation
rho_dist, p_dist = stats.spearmanr(dist_stats['listing_count'], dist_stats['med_revpar'])

# Left: Scatter
ax0 = axes[0]
ax0.scatter(dist_stats['listing_count'], dist_stats['med_revpar'],
             s=60, c=colors_set2[0], alpha=0.7, edgecolors='white', linewidth=1)

# Trend line
z = np.polyfit(dist_stats['listing_count'], dist_stats['med_revpar'], 1)
p_fit = np.poly1d(z)
x_line = np.linspace(dist_stats['listing_count'].min(), dist_stats['listing_count'].max(), 100)
ax0.plot(x_line, p_fit(x_line), 'r--', alpha=0.6, linewidth=1.5, label=f'추세선 (ρ={rho_dist:.3f})')

# Annotate top/bottom 5
top5 = dist_stats.nlargest(5, 'med_revpar')
bot5 = dist_stats.nsmallest(5, 'med_revpar')
for _, row in pd.concat([top5, bot5]).iterrows():
    ax0.annotate(row['district'].replace('-gu', '\ngu').replace('-', '\n'),
                 (row['listing_count'], row['med_revpar']),
                 fontsize=6.5, ha='center', va='bottom',
                 xytext=(0, 5), textcoords='offset points',
                 bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.6))

ax0.set_title(f'공급 규모 vs RevPAR\nSpearman ρ = {rho_dist:.3f} (p={p_dist:.3f})', fontsize=11, fontweight='bold')
ax0.set_xlabel('Active+Op 리스팅 수', fontsize=9)
ax0.set_ylabel('중위 TTM RevPAR (₩)', fontsize=9)
ax0.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'₩{int(x/1000):,}K'))
ax0.legend(fontsize=8)

# Right: Horizontal bar — top 15 districts by RevPAR
ax1 = axes[1]
top15 = dist_stats.nlargest(15, 'med_revpar').sort_values('med_revpar', ascending=True)

bar_colors_dist = []
for _, row in top15.iterrows():
    if row['listing_count'] > 500:
        bar_colors_dist.append(colors_set2[2])  # high supply
    elif row['listing_count'] > 200:
        bar_colors_dist.append(colors_set2[5])  # medium supply
    else:
        bar_colors_dist.append(colors_set2[0])  # low supply

bars1 = ax1.barh(range(len(top15)), top15['med_revpar'],
                  color=bar_colors_dist, alpha=0.85, edgecolor='white')
ax1.set_yticks(range(len(top15)))
ax1.set_yticklabels(top15['district'], fontsize=8)

for bar, (_, row) in zip(bars1, top15.iterrows()):
    ax1.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
             f'{fmt_krw_k(row["med_revpar"])} (n={row["listing_count"]:,})',
             va='center', fontsize=7)

legend_elements = [
    mpatches.Patch(color=colors_set2[0], alpha=0.8, label='저공급 (<200)'),
    mpatches.Patch(color=colors_set2[5], alpha=0.8, label='중공급 (200-500)'),
    mpatches.Patch(color=colors_set2[2], alpha=0.8, label='고공급 (>500)'),
]
ax1.legend(handles=legend_elements, fontsize=8, loc='lower right')
ax1.set_title('RevPAR 상위 15개 자치구', fontsize=11, fontweight='bold')
ax1.set_xlabel('중위 TTM RevPAR (₩)', fontsize=9)
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'₩{int(x/1000):,}K'))

plt.tight_layout()
plt.show()

print('=== H6: 자치구 공급 vs RevPAR ===')
print(f'Spearman ρ = {rho_dist:.3f}, p = {p_dist:.3f}')
print(f'자치구 수: {len(dist_stats)}개')
print(f'\n최고 RevPAR 자치구 Top 5:')
for _, row in dist_stats.nlargest(5, 'med_revpar').iterrows():
    print(f'  {row["district"]}: {fmt_krw(row["med_revpar"])} (n={row["listing_count"]:,})')
print(f'\n최저 RevPAR 자치구 Bottom 5:')
for _, row in dist_stats.nsmallest(5, 'med_revpar').iterrows():
    print(f'  {row["district"]}: {fmt_krw(row["med_revpar"])} (n={row["listing_count"]:,})')

In [ ]:
# H7: 관광지/문화시설 근접 숙소가 높은 RevPAR -- 단순 거리보다 POI 유형이 더 중요

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('H7: 인근 POI 유형 및 거리가 RevPAR에 미치는 영향 (Active+Op)', fontsize=13, fontweight='bold')

colors_set2 = sns.color_palette('Set2', 8)

df_ao_poi = df_ao[['nearest_poi_type_name', 'nearest_poi_dist_km', 'ttm_revpar']].dropna()

poi_stats = df_ao_poi.groupby('nearest_poi_type_name').agg(
    med_revpar=('ttm_revpar', 'median'),
    med_dist=('nearest_poi_dist_km', 'median'),
    count=('ttm_revpar', 'count')
).reset_index().sort_values('med_revpar', ascending=False)

# Left: Horizontal bar by POI type
ax0 = axes[0]
poi_sorted = poi_stats.sort_values('med_revpar', ascending=True)
bar_colors_poi = sns.color_palette('Set2', len(poi_sorted))

bars0 = ax0.barh(poi_sorted['nearest_poi_type_name'], poi_sorted['med_revpar'],
                  color=bar_colors_poi, alpha=0.85, edgecolor='white')

for bar, (_, row) in zip(bars0, poi_sorted.iterrows()):
    ax0.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
             f'{fmt_krw_k(row["med_revpar"])} (n={row["count"]:,})',
             va='center', fontsize=8)

ax0.set_title('POI 유형별 중위 RevPAR', fontsize=11, fontweight='bold')
ax0.set_xlabel('중위 TTM RevPAR (₩)', fontsize=9)
ax0.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'₩{int(x/1000):,}K'))
ax0.tick_params(axis='y', labelsize=9)

# Right: Scatter — POI distance vs RevPAR
ax1 = axes[1]
for i, (_, row) in enumerate(poi_stats.iterrows()):
    color = bar_colors_poi[i % len(bar_colors_poi)]
    ax1.scatter(row['med_dist'], row['med_revpar'],
                s=max(50, row['count'] / 10),
                c=[color], alpha=0.8, edgecolors='white', linewidth=1)
    ax1.annotate(row['nearest_poi_type_name'],
                 (row['med_dist'], row['med_revpar']),
                 fontsize=8, ha='center', va='bottom',
                 xytext=(0, 8), textcoords='offset points',
                 bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.6))

# Trend line
if len(poi_stats) >= 3:
    rho_poi, p_poi = stats.spearmanr(poi_stats['med_dist'], poi_stats['med_revpar'])
    ax1.set_xlabel(f'중위 POI 거리 (km) | Spearman ρ = {rho_poi:.3f}', fontsize=9)
else:
    ax1.set_xlabel('중위 POI 거리 (km)', fontsize=9)

ax1.set_title('POI 거리 vs RevPAR (버블=리스팅수)', fontsize=11, fontweight='bold')
ax1.set_ylabel('중위 TTM RevPAR (₩)', fontsize=9)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'₩{int(x/1000):,}K'))

plt.tight_layout()
plt.show()

print('=== H7: POI 유형별 RevPAR ===')
for _, row in poi_stats.iterrows():
    print(f'{row["nearest_poi_type_name"]:12s}: RevPAR {fmt_krw(row["med_revpar"]):>12s}, 중위거리 {row["med_dist"]:.2f}km (n={row["count"]:,})')

rho_poi_dist, p_poi_dist = stats.spearmanr(df_ao_poi['nearest_poi_dist_km'], df_ao_poi['ttm_revpar'])
print(f'\nPOI 거리 vs RevPAR (리스팅 레벨): Spearman ρ = {rho_poi_dist:.3f}, p = {p_poi_dist:.2e}')

In [ ]:
# H8: 외국인 비율 높은 관광형 자치구가 RevPAR 최고 -- 수요 특성이 입지 선택의 핵심

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('H8+H9: 인구·Dormant 비율이 자치구 RevPAR에 미치는 영향', fontsize=13, fontweight='bold')

colors_set2 = sns.color_palette('Set2', 8)

# District-level stats
dist_full = df.groupby('district').agg(
    total_count=('ttm_revpar', 'count'),
    med_pop=('ttm_pop', 'median')
).reset_index()

dist_active = df_ao.groupby('district').agg(
    active_count=('ttm_revpar', 'count'),
    med_revpar_ao=('ttm_revpar', 'median')
).reset_index()

dist_dormant = df[df['refined_status'] == 'Dormant'].groupby('district').agg(
    dormant_count=('ttm_revpar', 'count')
).reset_index()

dist_h89 = dist_full.merge(dist_active, on='district', how='left')
dist_h89 = dist_h89.merge(dist_dormant, on='district', how='left')
dist_h89['dormant_ratio'] = dist_h89['dormant_count'] / dist_h89['total_count']
dist_h89 = dist_h89.dropna(subset=['med_revpar_ao', 'med_pop', 'dormant_ratio'])

# Left (H8): Population vs RevPAR
ax0 = axes[0]
ax0.scatter(dist_h89['med_pop'], dist_h89['med_revpar_ao'],
             s=dist_h89['active_count'].fillna(50) / 5,
             c=colors_set2[0], alpha=0.7, edgecolors='white', linewidth=1)

# Annotate notable districts
notable_h8 = ['Mapo-gu', 'Gangnam-gu', 'Jung-gu', 'Jongno-gu', 'Dobong-gu', 'Nowon-gu']
for _, row in dist_h89[dist_h89['district'].isin(notable_h8)].iterrows():
    ax0.annotate(row['district'].replace('-gu', ''),
                 (row['med_pop'], row['med_revpar_ao']),
                 fontsize=8, ha='center', va='bottom',
                 xytext=(0, 6), textcoords='offset points',
                 bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.6))

# Trend line
rho_pop, p_pop = stats.spearmanr(dist_h89['med_pop'], dist_h89['med_revpar_ao'])
z_pop = np.polyfit(dist_h89['med_pop'].dropna(), dist_h89['med_revpar_ao'].dropna(), 1)
p_pop_fit = np.poly1d(z_pop)
x_range_pop = np.linspace(dist_h89['med_pop'].min(), dist_h89['med_pop'].max(), 100)
ax0.plot(x_range_pop, p_pop_fit(x_range_pop), 'r--', alpha=0.5, linewidth=1.5,
          label=f'추세선 (ρ={rho_pop:.3f})')
ax0.legend(fontsize=8)

ax0.set_title(f'H8: 인구 vs RevPAR\nSpearman ρ = {rho_pop:.3f}, p = {p_pop:.3f}', fontsize=11, fontweight='bold')
ax0.set_xlabel('중위 자치구 인구 (명)', fontsize=9)
ax0.set_ylabel('Active+Op 중위 TTM RevPAR (₩)', fontsize=9)
ax0.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{int(x/10000):,}만'))
ax0.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'₩{int(x/1000):,}K'))

# Right (H9): Dormant ratio vs RevPAR
ax1 = axes[1]
sc = ax1.scatter(dist_h89['dormant_ratio'], dist_h89['med_revpar_ao'],
                  s=dist_h89['active_count'].fillna(50) / 5,
                  c=dist_h89['dormant_ratio'], cmap='RdYlGn_r',
                  alpha=0.8, edgecolors='white', linewidth=1)
plt.colorbar(sc, ax=ax1, label='Dormant 비율', shrink=0.8)

# Annotate
notable_h9 = ['Mapo-gu', 'Gangdong-gu', 'Dongjak-gu', 'Seodaemun-gu', 'Gangnam-gu']
for _, row in dist_h89[dist_h89['district'].isin(notable_h9)].iterrows():
    ax1.annotate(row['district'].replace('-gu', ''),
                 (row['dormant_ratio'], row['med_revpar_ao']),
                 fontsize=8, ha='center', va='bottom',
                 xytext=(0, 6), textcoords='offset points',
                 bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.6))

# Trend line
rho_dorm, p_dorm = stats.spearmanr(dist_h89['dormant_ratio'], dist_h89['med_revpar_ao'])
z_dorm = np.polyfit(dist_h89['dormant_ratio'], dist_h89['med_revpar_ao'], 1)
p_dorm_fit = np.poly1d(z_dorm)
x_range_dorm = np.linspace(dist_h89['dormant_ratio'].min(), dist_h89['dormant_ratio'].max(), 100)
ax1.plot(x_range_dorm, p_dorm_fit(x_range_dorm), 'r--', alpha=0.5, linewidth=1.5,
          label=f'추세선 (ρ={rho_dorm:.3f})')
ax1.legend(fontsize=8)

ax1.set_title(f'H9: Dormant 비율 vs RevPAR\nSpearman ρ = {rho_dorm:.3f}, p = {p_dorm:.3f}', fontsize=11, fontweight='bold')
ax1.set_xlabel('자치구 Dormant 비율', fontsize=9)
ax1.set_ylabel('Active+Op 중위 TTM RevPAR (₩)', fontsize=9)
ax1.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'₩{int(x/1000):,}K'))

plt.tight_layout()
plt.show()

print('=== H8: 인구 vs RevPAR ===')
print(f'Spearman ρ = {rho_pop:.3f}, p = {p_pop:.3f}')

print('\n=== H9: Dormant 비율 vs RevPAR ===')
print(f'Spearman ρ = {rho_dorm:.3f}, p = {p_dorm:.3f}')
print(f'\n자치구별 Dormant 비율 상세:')
print(dist_h89[['district', 'dormant_ratio', 'med_revpar_ao']].sort_values('dormant_ratio', ascending=False).head(8).to_string(index=False))

In [ ]:
# 영향도 x 실행 용이성 2x2 매트릭스 -- 호스트 관점의 우선순위 액션 시각화

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('RevPAR 최대화 액션 우선순위 매트릭스', fontsize=13, fontweight='bold')

colors_set2 = sns.color_palette('Set2', 8)

# Action matrix data
action_data = {
    '가설': ['H1\n슈퍼호스트', 'H2\n숙소유형', 'H3\n콘텐츠품질', 'H4\n최소숙박일', 'H5\n추가요금',
             'H6\n지역선택', 'H7\nPOI입지', 'H9\n시장건전성'],
    '실행용이성': [3, 2, 5, 5, 4, 1, 1, 2],  # 1=어려움, 5=쉬움
    'RevPAR영향력': [5, 4, 3, 4, 3, 4, 3, 4],  # 1=낮음, 5=높음
    '비용': [2, 5, 1, 1, 1, 5, 4, 3]  # 1=저비용, 5=고비용 (낮을수록 좋음)
}
df_action = pd.DataFrame(action_data)

# Left: Heatmap
ax0 = axes[0]
heatmap_data = df_action.set_index('가설')[['실행용이성', 'RevPAR영향력', '비용']].T

# For heatmap: invert cost (lower cost = higher score)
heatmap_display = heatmap_data.copy()
heatmap_display.loc['비용'] = 6 - heatmap_display.loc['비용']  # invert: 5=good (low cost)

sns.heatmap(heatmap_display, ax=ax0, cmap='RdYlGn', vmin=1, vmax=5,
             annot=True, fmt='.0f', cbar_kws={'label': '점수 (높을수록 좋음)'},
             linewidths=0.5, linecolor='white')
ax0.set_title('액션 평가 매트릭스\n(비용: 역척도 — 높을수록 저비용)', fontsize=11, fontweight='bold')
ax0.set_xlabel('')
ax0.tick_params(axis='x', labelsize=8, rotation=45)
ax0.tick_params(axis='y', labelsize=9, rotation=0)

# Right: Bubble scatter
ax1 = axes[1]

# Color by priority quadrant
bubble_colors = []
for _, row in df_action.iterrows():
    ease = row['실행용이성']
    impact = row['RevPAR영향력']
    if ease >= 3 and impact >= 4:
        bubble_colors.append(colors_set2[0])  # high priority: easy + high impact
    elif ease < 3 and impact >= 4:
        bubble_colors.append(colors_set2[1])  # strategic: hard but high impact
    elif ease >= 3 and impact < 4:
        bubble_colors.append(colors_set2[5])  # quick win: easy but lower impact
    else:
        bubble_colors.append(colors_set2[2])  # low priority

# Bubble size: inverse of cost (low cost = large bubble)
bubble_sizes = [(6 - c) * 120 for c in df_action['비용']]

for i, row in df_action.iterrows():
    ax1.scatter(row['실행용이성'], row['RevPAR영향력'],
                s=bubble_sizes[i], c=[bubble_colors[i]], alpha=0.8,
                edgecolors='white', linewidth=1.5, zorder=5)
    ax1.annotate(row['가설'], (row['실행용이성'], row['RevPAR영향력']),
                 fontsize=8, ha='center', va='center', fontweight='bold',
                 color='white' if bubble_sizes[i] > 300 else 'black')

# Quadrant lines
ax1.axhline(3.5, color='gray', linestyle='--', alpha=0.5, linewidth=1)
ax1.axvline(3.5, color='gray', linestyle='--', alpha=0.5, linewidth=1)

# Quadrant labels
ax1.text(0.75, 4.75, '전략 과제\n(어렵지만 임팩트↑)', fontsize=7.5, color='gray', alpha=0.8, ha='center')
ax1.text(4.5, 4.75, '즉시 실행\n우선순위 1', fontsize=7.5, color='green', fontweight='bold', ha='center')
ax1.text(0.75, 1.5, '장기 검토', fontsize=7.5, color='gray', alpha=0.8, ha='center')
ax1.text(4.5, 1.5, '빠른 승리\n(쉽지만 작은 임팩트)', fontsize=7.5, color='orange', ha='center')

legend_elements_act = [
    mpatches.Patch(color=colors_set2[0], alpha=0.8, label='즉시 실행 (고임팩트+쉬움)'),
    mpatches.Patch(color=colors_set2[1], alpha=0.8, label='전략 과제 (고임팩트+어려움)'),
    mpatches.Patch(color=colors_set2[5], alpha=0.8, label='빠른 승리 (쉽지만 중임팩트)'),
]
ax1.legend(handles=legend_elements_act, fontsize=7, loc='lower left')
ax1.set_title('액션 우선순위 버블 차트\n(버블 크기 = 저비용 정도)', fontsize=11, fontweight='bold')
ax1.set_xlabel('실행 용이성 (1=어려움, 5=쉬움)', fontsize=9)
ax1.set_ylabel('RevPAR 영향력 (1=낮음, 5=높음)', fontsize=9)
ax1.set_xlim(0.5, 5.5)
ax1.set_ylim(0.5, 5.5)

plt.tight_layout()
plt.show()

print('=== 액션 우선순위 매트릭스 ===')
df_action['종합점수'] = df_action['실행용이성'] + df_action['RevPAR영향력'] + (6 - df_action['비용'])
priority_df = df_action.sort_values('종합점수', ascending=False)
print(priority_df[['가설', '실행용이성', 'RevPAR영향력', '비용', '종합점수']].to_string(index=False))

## 4. 임원진 종합 요약 (Executive Summary)

---

### 핵심 발견 3가지

**1. 시장 이중 구조 — Dormant 54.3%가 시장 왜곡의 핵심**

전체 32,061개 리스팅 중 17,399개(54.3%)가 Dormant 상태로 사실상 비활성화되어 있다. 이로 인해 전체 중위 RevPAR(₩8,868)와 Active+Operating 중위 RevPAR(₩47,850) 사이에 **5.4배** 격차가 발생한다. 이 구조적 문제가 해소되지 않으면 플랫폼 전체 RevPAR 개선은 어렵다.

**2. 슈퍼호스트 지위가 가장 강력한 단일 RevPAR 레버 (+83.1%)**

Active+Operating 내에서 슈퍼호스트는 비슈퍼호스트 대비 중위 RevPAR ₩79,087 vs ₩43,201로 **83.1% 프리미엄**을 기록한다(Mann-Whitney p<0.001). 슈퍼호스트 달성의 핵심은 평점(4.8+), 응답률, 취소율이며 — 이 세 가지는 모두 호스트가 통제 가능한 변수다.

**3. 최소숙박일 2-3박이 RevPAR 최적점 — 단순 단기 편향은 ADR을 희생**

RevPAR = ADR × 점유율 구조에서 점유율이 Spearman ρ=0.955로 압도적 드라이버다. **RevPAR 최적 = 3박(₩58,540), 점유율 최적 = 4-7박(0.597)** — 두 최적점이 불일치하므로 호스트는 목적(점유율 vs ADR)에 따라 설정을 선택해야 한다. 또한 **Perfectionism Paradox**: 평점 5.0 목표보다 4.85~4.95 구간(점유율 0.622)이 실질 RevPAR에 유리함.

---

### 호스트를 위한 Top 5 액션 (ROI 순)

| 순위 | 액션 | 기대 RevPAR 증가 | 실행 비용 | 구현 기간 |
|------|------|-----------------|-----------|----------|
| **1** | **최소숙박일 1-2박으로 단축** | 점유율 최대화 → RevPAR 상위 구간 진입 | 무료 | 즉시 |
| **2** | **슈퍼호스트 기준 달성** (평점 4.8+, 응답률 90%+, 취소 1% 미만) | +83.1% (₩43K → ₩79K) | 노력 비용 | 3~6개월 |
| **3** | **사진 21-35장 업로드** | 검색 노출 + 전환율 개선 | 사진 촬영비 | 1주 |
| **4** | **평점 4.8 이상 관리** | 임계점 초과 시 RevPAR 뚜렷한 도약 | 서비스 개선 | 지속 |
| **5** | **소형 숙소(1-6인) 추가 게스트 요금 설정** | 5천~3만원 범위에서 제한적 효과 (⚠️ 7인+ 대형은 RevPAR -56% 역효과) | 무료 | 즉시 |

---

### 플랫폼을 위한 Top 3 전략 (자치구 단위)

**전략 1: 마포구 공급 과잉 관리**

마포구는 전체 리스팅의 21%(6,699개)를 차지한다. **주목**: 마포구는 공급 1위이면서도 RevPAR 상위권을 유지하고 있어, 단순 공급-RevPAR 역관계 가설(H6)과 일부 상충된다. 추가 세분화 분석(슈퍼호스트 비율, POI 유형) 필요. 플랫폼 차원에서 마포구 신규 호스트 온보딩 시 RevPAR 현실을 명시하고, 차별화 전략(슈퍼호스트, 사진 품질, 즉시예약)을 강조하는 맞춤형 가이드가 필요하다.

**전략 2: Dormant → Active 전환 프로그램**

17,399개 Dormant 리스팅 중 10%만 Active+Operating으로 전환되어도 약 1,740개 신규 수익 창출 리스팅이 된다. 이를 위해 플랫폼은 Dormant 호스트 대상 ① 가격 최적화 도구 제공 ② 재활성화 인센티브 ③ 슈퍼호스트 육성 캠페인을 운영해야 한다.

**전략 3: RevPAR 상위 자치구 집중 마케팅**

중구(Jung-gu), 종로구(Jongno-gu), 강남구(Gangnam-gu) 등 RevPAR 상위 자치구는 저공급-고RevPAR 구조를 보인다. 이 지역 신규 호스트 유치 및 기존 호스트 업그레이드 지원이 플랫폼 전체 RevPAR 상승에 직결된다.

---

### 시장 위험 요인

1. **Dormant 비율 54.3%**: 시장 건전성의 핵심 위험. Dormant 리스팅은 고객 경험을 저하시키고 플랫폼 신뢰도를 낮춘다.
2. **마포구 공급 집중**: 공급 과잉으로 인한 RevPAR 압박이 지속될 경우 호스트 이탈 위험이 증가한다.
3. **규제 리스크**: 서울시의 단기임대 규제 강화(공유숙박 허가제 등)가 시장 구조 전반에 영향을 미칠 수 있다. TTM 기간(2024-10~2025-09) 내 정책 변화 모니터링이 필요하다.
4. **외국인 수요 변동성**: 환율, 관광 트렌드 변화에 따른 수요 변동이 ADR에 영향을 미친다.

---

### 다음 단계

본 EDA를 기반으로 **`notebooks/03_host_modeling.ipynb`**에서 다음 모델링 작업이 수행된다:

1. **호스트 관점 예측 모델**: LightGBM + RandomForest + Ridge (5-Fold CV, RANDOM_SEED=42)
   - 입력: 호스트 통제 가능 변수 16개 (슈퍼호스트, 즉시예약, 사진수, 평점, 최소숙박일 등)
   - 타겟: `log1p(ttm_revpar)` → 역변환으로 원단위 해석
   - 평가: R², MAE, RMSE, MAPE
   - SHAP 분석으로 변수 기여도 정량화

2. **자치구 단위 모델**: Ridge + LeaveOneOut CV (n=25 소표본)
   - 입력: 자치구 집계 통계 + POI + 인구 + Dormant 비율
   - 목적: 자치구별 RevPAR 예측 및 군집 특성 해석

---

*분석 완료일: 2026-02-20 | 데이터 기간: TTM 2024-10-01 ~ 2025-09-30 | 작성: Seoul Airbnb RevPAR Analysis Project*

In [ ]:
# 전체 가설 검증 결과 요약 출력 -- 보고서 핵심 수치로 활용

print('=' * 60)
print('  서울 에어비앤비 RevPAR 최대화 분석 — 최종 통계 요약')
print('=' * 60)

print('\n[시장 현황]')
print(f'  전체 리스팅:           {len(df):,}개')
print(f'  Active+Operating:      {len(df_ao):,}개 ({len(df_ao)/len(df)*100:.1f}%)')
dormant_n = (df['refined_status'] == 'Dormant').sum()
print(f'  Dormant:               {dormant_n:,}개 ({dormant_n/len(df)*100:.1f}%)')
print(f'  TTM RevPAR 전체 중위:  {fmt_krw(df["ttm_revpar"].median())}')
print(f'  TTM RevPAR AO 중위:    {fmt_krw(df_ao["ttm_revpar"].median())}')
print(f'  TTM RevPAR AO 평균:    {fmt_krw(df_ao["ttm_revpar"].mean())}')

print('\n[H1: 슈퍼호스트]')
sh_med = df_ao[df_ao['superhost']==True]['ttm_revpar'].median()
nsh_med = df_ao[df_ao['superhost']==False]['ttm_revpar'].median()
print(f'  슈퍼호스트 중위 RevPAR:  {fmt_krw(sh_med)}')
print(f'  일반호스트 중위 RevPAR:  {fmt_krw(nsh_med)}')
print(f'  프리미엄:                {sh_med/nsh_med*100-100:.1f}%')

print('\n[H2: 숙소 유형별 RevPAR (Active+Op)]')
for rt in ['entire_home', 'private_room', 'hotel_room', 'shared_room']:
    sub = df_ao[df_ao['room_type']==rt]['ttm_revpar']
    if len(sub) > 0:
        print(f'  {rt:15s}: {fmt_krw(sub.median())} (n={len(sub):,})')

print('\n[H5: 추가요금 정책 (guests≥4)]')
df_ao_lg = df_ao[df_ao['guests']>=4]
f0 = df_ao_lg[df_ao_lg['extra_guest_fee_policy']==0]['ttm_revpar'].median()
f1 = df_ao_lg[df_ao_lg['extra_guest_fee_policy']==1]['ttm_revpar'].median()
print(f'  요금 없음: {fmt_krw(f0)} | 요금 있음: {fmt_krw(f1)} | 차이: {fmt_krw(f1-f0)}')

print('\n[H6: 공급 집중 자치구]')
top3_dist = df['district'].value_counts().head(3)
for dist, cnt in top3_dist.items():
    print(f'  {dist}: {cnt:,}개 ({cnt/len(df)*100:.1f}%)')

print('\n[저장된 시각화 파일]')
import glob
saved_figs = sorted(glob.glob(f'{REPORTS_DIR}/fig_07_*.png'))
for f in saved_figs:
    print(f'  {f}')

print('\n분석 완료.')